# Exploratory Data Analysis for Project Defaulter Early Flagging

## A. Transaction Data Wrangling

### 1. Reading the Data

In [100]:
# Import all the required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [101]:
# Configure the output directories
files_dir = '/kaggle/working/data'
os.makedirs(files_dir, exist_ok=True)

def save_csv(frame, filename):
    files_dir = '/kaggle/working/data'
    os.makedirs(files_dir, exist_ok=True)

    full_path = os.path.join(files_dir, filename)
    frame.to_csv(full_path, index=False)
    print(f"{filename} saved to: {full_path}")

In [102]:
# Load the dataset (low_memory=false, since it was reading a columntype wrong in an attempt without it.)
dft = pd.read_csv('/kaggle/input/datasets/parasharmanu/the-berka-filtered/trans.csv', low_memory=False)

In [103]:
dft.shape

(1056320, 10)

> ##### *The data contains more than a million rows, and 10 attributes.* 

In [104]:
dft.head()

,trans_id,account_id,date,type,operation,amount,balance,k_symbol,bank,account
0,695247,2378,930101,PRIJEM,VKLAD,700.0,700.0,NaN,NaN,NaN
1,171812,576,930101,PRIJEM,VKLAD,900.0,900.0,NaN,NaN,NaN
2,207264,704,930101,PRIJEM,VKLAD,1000.0,1000.0,NaN,NaN,NaN
3,1117247,3818,930101,PRIJEM,VKLAD,600.0,600.0,NaN,NaN,NaN
4,579373,1972,930102,PRIJEM,VKLAD,400.0,400.0,NaN,NaN,NaN


> ##### *The date column is integer data type.* 

> ##### *The string data is in Czech Language.* 

In [105]:
dft.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1056320 entries, 0 to 1056319
Data columns (total 10 columns):
 #   Column      Non-Null Count    Dtype  
---  ------      --------------    -----  
 0   trans_id    1056320 non-null  int64  
 1   account_id  1056320 non-null  int64  
 2   date        1056320 non-null  int64  
 3   type        1056320 non-null  object 
 4   operation   873206 non-null   object 
 5   amount      1056320 non-null  float64
 6   balance     1056320 non-null  float64
 7   k_symbol    574439 non-null   object 
 8   bank        273508 non-null   object 
 9   account     295389 non-null   float64
dtypes: float64(3), int64(3), object(4)
memory usage: 80.6+ MB


> ##### *'bank' and 'account' colums containing null values makes sense. Since, these are only used in bank-to-bank transfers.* 

In [106]:
dft.describe()

,trans_id,account_id,date,amount,balance,account
count,1.056320e+06,1.056320e+06,1.056320e+06,1.056320e+06,1.056320e+06,2.953890e+05
mean,1.335311e+06,2.936867e+03,9.656748e+05,5.924146e+03,3.851833e+04,4.567092e+07
std,1.227487e+06,2.477345e+03,1.394535e+04,9.522735e+03,2.211787e+04,3.066340e+07
min,1.000000e+00,1.000000e+00,9.301010e+05,0.000000e+00,-4.112570e+04,0.000000e+00
25%,4.302628e+05,1.204000e+03,9.601160e+05,1.359000e+02,2.240250e+04,1.782858e+07
50%,8.585065e+05,2.434000e+03,9.704100e+05,2.100000e+03,3.314340e+04,4.575095e+07
75%,2.060979e+06,3.660000e+03,9.802280e+05,6.800000e+03,4.960362e+04,7.201341e+07
max,3.682987e+06,1.138200e+04,9.812310e+05,8.740000e+04,2.096370e+05,9.999420e+07


> ##### *The median of amount is 2,100, while the maximum is 87,400. Clearly implies that the transaction amounts are skewed towards higher amounts. Clears with the logic that balances are also skewed, median being 33,143, and maximum being 2,09,637.* 

> ##### *Minimum bank balance is negative 41,125. States that there are accounts with substantial negative balances.* 

In [107]:
dft.nunique()

trans_id      1056320
account_id       4500
date             2191
type                3
operation           5
amount          40400
balance        542739
k_symbol            8
bank               13
account          7665
dtype: int64

> ##### *There are only 4500 accounts dealing with about a million transactions in this data.* 

In [108]:
dft.isnull().sum()

trans_id           0
account_id         0
date               0
type               0
operation     183114
amount             0
balance            0
k_symbol      481881
bank          782812
account       760931
dtype: int64

> ##### *There are about 8 lakh missing values in banks and accounts. Implies that most of the transfers are internals within the same bank.* 

### 2. Transforming the Data

> ##### *Converting date column to datetime.* 

In [109]:
# Converting the date column to datetime columntype.
dft['date'] = pd.to_datetime('19' + dft['date'].astype(str), format='%Y%m%d')
dft['date'].head(10)

0   1993-01-01
1   1993-01-01
2   1993-01-01
3   1993-01-01
4   1993-01-02
5   1993-01-02
6   1993-01-03
7   1993-01-03
8   1993-01-03
9   1993-01-03
Name: date, dtype: datetime64[ns]

> ##### *Mapping the Czech strings to English.* 

In [110]:
# Listing all the unique strings in operation column.
dft['operation'].unique()

array(['VKLAD', 'PREVOD Z UCTU', 'VYBER', nan, 'PREVOD NA UCET',
       'VYBER KARTOU'], dtype=object)

In [111]:
# Listing all the unique strings in type column.
dft['type'].unique()

array(['PRIJEM', 'VYDAJ', 'VYBER'], dtype=object)

In [112]:
# Listing all the unique strings in k_symbol column.
dft['k_symbol'].unique()

array([nan, 'DUCHOD', 'UROK', 'SIPO', 'SLUZBY', ' ', 'POJISTNE',
       'SANKC. UROK', 'UVER'], dtype=object)

In [ ]:
# Mapping the Czech strings
type_map = {
    'PRIJEM': 'Credit', 
    'VYDAJ': 'Debit',
    'VYBER': 'Withdrawal'
}

operation_map = {
    'VKLAD': 'Cash Deposit', 
    'PREVOD Z UCTU': 'Bank Transfer Inbound', 
    'VYBER': 'Cash Withdrawal',
    'PREVOD NA UCET': 'Bank Transfer Outbound',
    'VYBER KARTOU': 'Credit Card Withdrawal'
}
k_symbol_map = {
    'UROK':'Interest Credited',
    'SLUZBY':'Service Bill Payment',
    'SIPO':'Household Payment',
    'UVER':'Loan repayment',
    'POJISTNE':'Insurance premium',
    'DUCHOD':'Pension received',
    'SANKC. UROK':'Sanction Interest'
}

dft['type'] = dft['type'].map(type_map)
dft['operation'] = dft['operation'].map(operation_map).fillna('Other')
dft['k_symbol'] = dft['k_symbol'].map(k_symbol_map).fillna('NA')

dft.head(40)

,trans_id,account_id,date,type,operation,amount,balance,k_symbol,bank,account
0,695247,2378,1993-01-01,Credit,Cash Deposit,700.0,700.0,NA,NaN,NaN
1,171812,576,1993-01-01,Credit,Cash Deposit,900.0,900.0,NA,NaN,NaN
2,207264,704,1993-01-01,Credit,Cash Deposit,1000.0,1000.0,NA,NaN,NaN
3,1117247,3818,1993-01-01,Credit,Cash Deposit,600.0,600.0,NA,NaN,NaN
4,579373,1972,1993-01-02,Credit,Cash Deposit,400.0,400.0,NA,NaN,NaN
5,771035,2632,1993-01-02,Credit,Cash Deposit,1100.0,1100.0,NA,NaN,NaN
6,452728,1539,1993-01-03,Credit,Cash Deposit,600.0,600.0,NA,NaN,NaN
7,725751,2484,1993-01-03,Credit,Cash Deposit,1100.0,1100.0,NA,NaN,NaN
8,497211,1695,1993-01-03,Credit,Cash Deposit,200.0,200.0,NA,NaN,NaN
9,232960,793,1993-01-03,Credit,Cash Deposit,800.0,800.0,NA,NaN,NaN


In [114]:
dft.columns.tolist()

['trans_id',
 'account_id',
 'date',
 'type',
 'operation',
 'amount',
 'balance',
 'k_symbol',
 'bank',
 'account']

In [115]:
# Exporting the cleaned data to csv for visualization
save_csv(dft, 'transactions.csv')

transactions.csv saved to: /kaggle/working/data/transactions.csv


## B. Loan Data Wrangling

### 1. Reading the Data

In [116]:
# Read the loan data. Need to add a specified delimiter since the csv uses ';' as one.
dfl = pd.read_csv('/kaggle/input/datasets/parasharmanu/the-berka-filtered/loan.csv', sep=';', low_memory=False)

In [117]:
dfl.shape

(682, 7)

> ##### *There are only 682 rows, as compared to a million in the transactions data.* 

In [118]:
dfl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 682 entries, 0 to 681
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   loan_id     682 non-null    int64  
 1   account_id  682 non-null    int64  
 2   date        682 non-null    int64  
 3   amount      682 non-null    int64  
 4   duration    682 non-null    int64  
 5   payments    682 non-null    float64
 6   status      682 non-null    object 
dtypes: float64(1), int64(5), object(1)
memory usage: 37.4+ KB


> ##### *No missing values, all the correct data types. No need of any adjustments here.* 

In [119]:
dfl.head()

,loan_id,account_id,date,amount,duration,payments,status
0,5314,1787,930705,96396,12,8033.0,B
1,5316,1801,930711,165960,36,4610.0,A
2,6863,9188,930728,127080,60,2118.0,A
3,5325,1843,930803,105804,36,2939.0,A
4,7240,11013,930906,274740,60,4579.0,A


> ##### *The date column is integer, again.* 

In [120]:
dfl.isnull().sum()

loan_id       0
account_id    0
date          0
amount        0
duration      0
payments      0
status        0
dtype: int64

In [121]:
# Verifying if the payments column actually needs the float datatype
fractional_rows = dfl[dfl['payments'] % 1 != 0]
print(f'Total decimal rows: {len(fractional_rows)}')

Total decimal rows: 0


> ##### *No actual decimal numbers in floats. Safe to convert the columntype to integer easier calculations and visualizations.* 

In [122]:
dfl['payments'] = dfl['payments'].astype('int64')
dfl['payments'].head(10)

0    8033
1    4610
2    2118
3    2939
4    4579
5    3660
6    4399
7    7281
8    3217
9    4876
Name: payments, dtype: int64

In [123]:
# Verifying if there are more encodings than 2 in status column.
dfl['status'].unique()

array(['B', 'A', 'C', 'D'], dtype=object)

### 2. Transforming the Data

> ##### *Converting date column to datetime.* 

In [124]:
# Converting the date column to datetime columntype.
dfl['date'] = pd.to_datetime('19' + dfl['date'].astype(str), format='%Y%m%d')
dfl['date'].head(10)

0   1993-07-05
1   1993-07-11
2   1993-07-28
3   1993-08-03
4   1993-09-06
5   1993-09-13
6   1993-09-15
7   1993-09-24
8   1993-10-13
9   1993-11-04
Name: date, dtype: datetime64[ns]

In [125]:
# Adding an end-date column specifying the loan end date
dfl['end_date'] = dfl.apply(
    lambda row: row['date'] + pd.DateOffset(months=int(row['duration'])), 
    axis=1
)
dfl['end_date'].head(10)

0   1994-07-05
1   1996-07-11
2   1998-07-28
3   1996-08-03
4   1998-09-06
5   1995-09-13
6   1994-09-15
7   1995-09-24
8   1997-10-13
9   1995-11-04
Name: end_date, dtype: datetime64[ns]

In [126]:
# Translating the status codes to boolean columns
status_map = {
    'A': (1,0),
    'B': (1,1),
    'C': (0,0),
    'D': (0,1)
}

dfl[['running', 'defaulter']] = pd.DataFrame(dfl['status'].map(status_map).tolist(), index=dfl.index).fillna(0).astype(int)

dfl[['running', 'defaulter']].head(100)

dfl[dfl['running'] == 0]

,loan_id,account_id,date,amount,duration,payments,status,end_date,running,defaulter
23,5170,1071,1994-01-20,253200,60,4220,C,1999-01-20,0,0
30,6087,5313,1994-02-27,300660,60,5011,C,1999-02-27,0,0
38,7055,10079,1994-04-06,167100,60,2785,C,1999-04-06,0,0
39,6103,5385,1994-04-07,149340,60,2489,C,1999-04-07,0,0
42,6696,8321,1994-05-09,89040,60,1484,C,1999-05-09,0,0
...,...,...,...,...,...,...,...,...,...,...
677,4989,105,1998-12-05,352704,48,7348,C,2002-12-05,0,0
678,5221,1284,1998-12-05,52512,12,4376,C,1999-12-05,0,0
679,6402,6922,1998-12-06,139488,24,5812,C,2000-12-06,0,0
680,5346,1928,1998-12-06,55632,24,2318,C,2000-12-06,0,0


In [127]:
# Deleting the status column
dfl = dfl.drop('status', axis=1)

In [128]:
# Verifying the status conditions
filtered_dfl = (dfl[(dfl['end_date'].dt.year > 1998) & (dfl['running'] == 1)])
filtered_dfl.head(10)


,loan_id,account_id,date,amount,duration,payments,end_date,running,defaulter


> ##### *Confirmed that there are no incorrect dates.* 

In [129]:
# Rename the columns for easier understanding
dfl.rename(columns={'date':'start_date','amount':'total_amount','payments':'installment'}, inplace=True)

In [130]:
dfl.columns.tolist()

['loan_id',
 'account_id',
 'start_date',
 'total_amount',
 'duration',
 'installment',
 'end_date',
 'running',
 'defaulter']

In [131]:
# Exporting the cleaned data to csv for visualization
save_csv(dfl, 'loans.csv')

loans.csv saved to: /kaggle/working/data/loans.csv
